### Importing Python packages

In [ ]:
import pandas as pd
import geopandas as gpd
from sklearn.ensemble import RandomForestRegressor
import rasterio
from shapely.geometry import Point
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import numpy as np
import rioxarray as rxr
import xarray as xr
import dask.array as da
import dask
import dask.dataframe as dd
import cupy as cp
import time
from joblib import load
from dask.diagnostics import ProgressBar
import tempfile
import logging
import shutil
from scipy.spatial.distance import cdist, pdist
from sklearn.base import TransformerMixin
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from dask.distributed import Client, LocalCluster
# To manually clear memory
import gc
gc.disable()  # Disable automatic garbage collection
# Then manually call gc.collect() only after processing large batches
def clear_memory():
    gc.collect()
    client.run(gc.collect)  # Run garbage collection on all workers

### Load training data using GeoPandas and convert it to an Xarray dataset

In [ ]:
# Load training data file
file = r"original_training_data.gpkg"

gdf = gpd.read_file(file, driver = "gpkg", geometry = 'geometry')

# Drop NA values from the dataset
gdf.dropna(inplace = True)
gdf.reset_index(drop=True, inplace=True)

print("Total number of data: ", gdf.shape)

# change the order of the variables in the dataset similar to the raster data; from slope ...; delete geometry
training_columns =  ['soc', 'clay', 'silt', 'sand', 'rock', 'slope', 'lsf', 'tri', 'twi', 'ndvi_spring', 'ndvi_summer', 'ndvi_autumn', 'bsi_spring', 'bsi_summer', 'bsi_autumn', 'drained', 'landuse_type']
gdf = gdf[training_columns]

training_df = xr.Dataset(gdf)

### Load testing raster data and Preprocessing data

In [ ]:
def preprocess_raster_data(file_path, band_names, chunk_size=(1000, 1000)):
    """
    Load and preprocess raster data with memory-efficient processing.
    Steps include renaming bands, handling missing data, recoding land use types,
    and filtering out artificial and water areas.
    
    Parameters:
        - file_path (str): Path to the raster file.
        - band_names (list): List of band names.
        - chunk_size (tuple): Size of chunks for lazy processing.
    
    Returns:
        - xarray.Dataset: The preprocessed raster dataset.
        - xarray.DataArray: The original raster data.
    """
    # Landuse recoding dictionary
    landuse_lookup = {
        **{i: 'grassland' for i in range(10100, 12001, 100)},
        **{i: 'wetland' for i in range(20100, 20701, 100)}, 
        **{i: 'forest' for i in range(30100, 31201, 100)},
        **{i: 'arable' for i in range(40100, 40501, 100)},
        **{i: 'coastal' for i in range(50100, 51501, 100)},
    }
    default_value = np.nan  # Default value for codes not in the landuse_lookup
    
    def recode_landuse(da):
        def recode_function(block):
            # Handle floating-point values by rounding and casting to integers
            block = np.rint(block).astype(int)
    
            # Preallocate the recoded array
            recoded = np.empty_like(block, dtype=object)
    
            # Apply the landuse mapping
            for key, value in landuse_lookup.items():
                recoded[block == key] = value
    
            # Assign default value for unmatched keys
            recoded[~np.isin(block, list(landuse_lookup.keys()))] = default_value
    
            return recoded
    
        recoded_data = xr.apply_ufunc(
            recode_function,
            da,
            input_core_dims=[[]],
            output_core_dims=[[]],
            vectorize=False,
            dask="parallelized",
            output_dtypes=[object],
        )
    
        return recoded_data
    
    # Load raster data using rioxarray with larger chunks
    original_raster_data = rxr.open_rasterio(file_path, masked=True, chunks=chunk_size)
    
    # Ensure the number of bands need to matche the band names
    if len(band_names) != original_raster_data.sizes['band']:
        raise ValueError("The number of band names does not match the number of raster bands!")
    
    # Convert the raster data to xarray.Dataset and rename bands
    raster_dataset = original_raster_data.to_dataset(dim='band')
    raster_dataset = raster_dataset.rename({idx + 1: var for idx, var in enumerate(band_names)})
    
    # Apply preprocessing: recoding landuse type
    landuse_band = raster_dataset['landuse_type']
    recoded_landuse_band = recode_landuse(landuse_band)
    raster_dataset['landuse_type'] = recoded_landuse_band
    
    # Replace -9999 with NaN before stacking
    raster_dataset = raster_dataset.where(raster_dataset != -9999, np.nan)
    
    # Process in tiles to avoid memory issues
    y_size, x_size = raster_dataset.y.size, raster_dataset.x.size
    tile_size = 100000 
    
    # Create empty list to store processed chunks
    processed_chunks = []
    
    # Process in tiles
    for y_start in range(0, y_size, tile_size):
        y_end = min(y_start + tile_size, y_size)
        for x_start in range(0, x_size, tile_size):
            x_end = min(x_start + tile_size, x_size)
            
            # Extract tile
            tile = raster_dataset.isel(y=slice(y_start, y_end), x=slice(x_start, x_end))
            
            # Stack dimensions for this tile
            stacked_tile = tile.stack(dim_0=('y', 'x'))
            
            # Drop NaN values
            filtered_tile = stacked_tile.dropna(dim='dim_0', how='any')
                
            # Store the processed tile
            processed_chunks.append(filtered_tile)
    
    # If there is no valid data; return None
    if not processed_chunks:
        return None, original_raster_data
        
    # Combine all processed tiles into one single dataframe
    final_raster_dataset = xr.concat(processed_chunks, dim= 'dim_0')
    
    return final_raster_dataset, original_raster_data

### Read Estonia Raster Data

In [ ]:
import os
os.environ["DASK_DISTRIBUTED__WORKER__MEMORY__TERMINATE"] = "0.95"  # Allow workers to use up to 95% of memory limit
os.environ["DASK_DISTRIBUTED__WORKER__MEMORY__PAUSE"] = "0.85"      # Pause at 85% to prevent crashes
os.environ["DASK_DISTRIBUTED__WORKER__MEMORY__TARGET"] = "0.8" # Target 80% for optimal performance

cluster = LocalCluster(n_workers=8, threads_per_worker=2, memory_limit='10GB')
client = Client(cluster)
print(f"Dashboard link: {client.dashboard_link}")

In [ ]:
# Start the timer
start_time = time.time()

# Define band names for the raster
band_names = ['clay', 'silt', 'sand', 'rock', 'slope', 'lsf', 'tri', 'twi', 'ndvi_spring', 'ndvi_summer', 'ndvi_autumn', 'bsi_spring', 'bsi_summer', 'bsi_autumn', 'drained', 'landuse_type']

# Path to the raster file
file_path = r"estonia_merged_raster.tif"

# Process with fixed function
preprocessed_raster, original_raster = preprocess_raster_data(file_path, band_names)

preprocessed_raster

### Estimating Threshold of Disimiliarity Index (DI) for Area of Applicability (AOA)

#### Functions for computation

In [ ]:
# Standardization of raster data
class CustomStandardScalerOptimized(TransformerMixin):
    def __init__(self, vars_to_standardize):
        self.vars_to_standardize = vars_to_standardize
        self.scaler = StandardScaler()
        
    def fit(self, dataset, y=None):
        # Check if we're working with a DataFrame (training) or xarray dataset (raster)
        if isinstance(dataset, pd.DataFrame):
            # For pandas DataFrame, we can use sklearn's StandardScaler directly
            # Extract features to standardize
            features = dataset[self.vars_to_standardize].values
            self.scaler.fit(features)
            return self
        
        # For xarray Dataset, use chunked computation
        means = []
        variances = []
        n_samples = 0
        
        for var in self.vars_to_standardize:
            # Process each variable's data in chunks
            data = dataset[var].data.flatten()
            
            # If data is already a dask array, ensure it's properly chunked
            if isinstance(data, da.Array):
                # Use smaller chunks for better memory management
                data = data.rechunk((min(data.shape[0] // 10, 1_000_000),))
                # Compute mean and variance for this variable
                chunk_mean = data.mean().compute()
                chunk_var = data.var().compute()
            else:
                # For numpy arrays or other data types
                chunk_mean = np.mean(data)
                chunk_var = np.var(data)
            
            chunk_n = data.size
            
            means.append(chunk_mean)
            variances.append(chunk_var)
            n_samples = chunk_n  # Assuming all variables have same number of samples
        
        # Store statistics in the scaler
        self.scaler.mean_ = np.array(means)
        self.scaler.var_ = np.array(variances)
        self.scaler.scale_ = np.sqrt(self.scaler.var_)
        self.scaler.n_samples_seen_ = n_samples
        
        return self
    
    def transform(self, dataset):
        # Check if we're working with a DataFrame (training) or xarray dataset (raster)
        if isinstance(dataset, pd.DataFrame):
            # For pandas DataFrame, transform using sklearn's StandardScaler
            transformed_values = self.scaler.transform(dataset[self.vars_to_standardize].values)
            result = dataset.copy()
            result[self.vars_to_standardize] = transformed_values
            return result
        
        # For xarray Dataset, transform each variable separately
        standardized = dataset.copy()
        
        # Process each variable separately to avoid memory issues
        for idx, var in enumerate(self.vars_to_standardize):
            # Get the data for this variable
            data = dataset[var].data
            
            # Apply standardization without loading everything into memory
            if isinstance(data, da.Array):
                chunk_size = min(data.shape[0] // 10, 10000)
            
                # If chunk_size is 0, set it to 1
                if chunk_size == 0:
                    chunk_size = 1  # Set chunk size to 1 if the calculated chunk size is 0
    
                data = data.rechunk(chunks=(chunk_size, *data.shape[1:]))
                
                # Apply standardization formula manually
                std_data = (data - self.scaler.mean_[idx]) / self.scaler.scale_[idx]
                
                # Create a new DataArray with the standardized data
                standardized[var] = xr.DataArray(
                    std_data, 
                    dims=dataset[var].dims, 
                    coords=dataset[var].coords
                )
            else:
                # Convert to dask array if it's not already
                dask_data = da.from_array(data, chunks=(min(data.shape[0] // 10, 1_000_000), *data.shape[1:]))
                std_data = (dask_data - self.scaler.mean_[idx]) / self.scaler.scale_[idx]
                
                standardized[var] = xr.DataArray(
                    std_data, 
                    dims=dataset[var].dims, 
                    coords=dataset[var].coords
                )
                
        return standardized


def convert_to_binary_columns(ds, variable_name, unique_categories):
    """
    Converts a categorical variable in an xarray Dataset into binary columns.
    """
    if variable_name not in ds:
        raise ValueError(f"The variable '{variable_name}' does not exist in the Dataset.")
        
    dummy_ds = xr.Dataset()
    for category in unique_categories:
        binary_variable_name = f"{variable_name}_{category}"
        dummy_ds[binary_variable_name] = xr.where(ds[variable_name] == category, 1, 0)

    ds = ds.drop_vars(variable_name)
    ds = xr.merge([ds, dummy_ds])

    return ds

def calculate_mean_pairwise_distances(dataset):
    """
    Calculate the mean of all unique pairwise distances (no duplicates, no self-pairs).
    
    Parameters:
        - dataset (xarray.Dataset): The dataset with weighted standardised variables.
    
    Returns:
        - float: Mean of all unique pairwise Euclidean distances.
    """
    data_array = dataset.to_array().T.values.astype(float)  # Convert to 2D float array
    distances = pdist(data_array, metric='euclidean')     
    mean_distance = distances.mean()
    return mean_distance


def compute_dissimilarity_indexes(X_weighted_train, mean_distance_for_training):
    dissimilarity_indexes = []
    kf = KFold(n_splits=5, shuffle=True, random_state=25) # set the value of random state: 25 is used here

    # Convert to array, avoiding unnecessary stacking
    X_train_array = X_weighted_train.to_array().values.T  # Ensure X_train_array is a 2D numpy array

    # Iterate over the KFold splits
    for train_idx, test_idx in kf.split(X_train_array):
        train_data = X_train_array[train_idx]
        test_data = X_train_array[test_idx]

        # For each test point, calculate the minimum distance to any point in the training data
        for test_point in test_data:
            distances = cdist([test_point], train_data)  # Compute pairwise distances
            min_distance = distances.min()  # Get the minimum distance
            di = min_distance / mean_distance_for_training  # Dissimilarity index
            dissimilarity_indexes.append(di)  # Store the dissimilarity index

    # Calculate the interquartile range (IQR) and remove outliers based on a threshold
    q1 = np.percentile(dissimilarity_indexes, 25)
    q3 = np.percentile(dissimilarity_indexes, 75)
    iqr = q3 - q1
    outlier_removed_maximum_di = (1.5 * iqr) + q3  # 1.5 * IQR rule for outliers

    return outlier_removed_maximum_di
    
def compute_and_identify_AOA(X_weighted_train, X_weighted_test, mean_distance_for_training, threshold_di, 
                                batch_size = 8000000):
    """
    Compute the dissimilarity index (DI) and Area of Applicability (AOA) for the test dataset relative to the training dataset using xarray and processing in optimised batches.
    
    Parameters:
    -----------
    X_weighted_train : xarray.Dataset
        Training dataset with features
    X_weighted_test : xarray.Dataset
        Testing dataset with features
    mean_distance_for_training : float
        Mean distance between training data points
    threshold_di : float
        Threshold for the dissimilarity index to determine AOA
    batch_size : int, optional
        Number of test points to process in each batch
    checkpoint_interval : int, optional
        Number of records to process before saving a checkpoint
        
    Returns:
    --------
    xarray.Dataset
        Test dataset with added columns for dissimilarity_index and aoa
    """
    start_total = time.time()
    logging.info(f"Starting AOA calculation with batch size: {batch_size}")
    
    # Determine common features between training and test datasets
    common_features = list(set(X_weighted_train.data_vars) & set(X_weighted_test.data_vars))
    common_features = [f for f in common_features if f not in ['dissimilarity_index', 'aoa']]
    logging.info(f"Common features: {common_features}")
    
    # Convert training data to numpy array for efficient distance calculation
    train_arrays = []
    for feature in common_features:
        # Make sure to compute the dask arrays to get numpy arrays
        train_arrays.append(X_weighted_train[feature].values)
    
    X_train_array = np.stack(train_arrays, axis=1)
    total_num_training = X_train_array.shape[0]
    logging.info(f"Training data shape: {X_train_array.shape}")
    
    # Initialize result dataset 
    test_size = X_weighted_test.sizes['dim_0']
    dissimilarity_indexes = np.zeros(test_size, dtype=np.float32)
    aoa_values = np.zeros(test_size, dtype=np.int32)
    
    # Process the test data in batches
    logging.info(f"Processing {test_size} test points in batches of {batch_size}")
    
    for batch_start in tqdm(range(0, test_size, batch_size), desc="Processing test batches"):
        batch_time_start = time.time()
        batch_end = min(batch_start + batch_size, test_size)
        batch_indices = slice(batch_start, batch_end)
        
        # Extract batch of test data
        batch_data = []
        for feature in common_features:
            # Load this batch of test data
            feature_data = X_weighted_test[feature].isel(dim_0=batch_indices).values
            batch_data.append(feature_data)
        
        X_test_batch = np.stack(batch_data, axis=1)
        
        # Process small sub-batches more efficiently (vectorized operations)
        sub_batch_size = 4000000  
        for sub_batch_start in range(0, X_test_batch.shape[0], sub_batch_size):
            sub_batch_end = min(sub_batch_start + sub_batch_size, X_test_batch.shape[0])
            
            # Extract sub batch
            X_sub_batch = X_test_batch[sub_batch_start:sub_batch_end]
            
            # Calculate distances from this sub-batch to all training points (vectorized)
            distances = cdist(X_sub_batch, X_train_array)
            
            # Find minimum distances and calculate dissimilarity indexes
            min_distances = np.min(distances, axis=1)
            di_values = min_distances / mean_distance_for_training
            
            # Determine AOA values (1 if within threshold, 0 if not)
            aoa_batch = (di_values <= threshold_di).astype(int)
            
            # Store results for this sub-batch
            idx_offset = batch_start + sub_batch_start
            dissimilarity_indexes[idx_offset:idx_offset + len(di_values)] = di_values
            aoa_values[idx_offset:idx_offset + len(di_values)] = aoa_batch
            
            # Clean up to save memory
            del distances, di_values, aoa_batch
        
        # Clean up batch data to save memory
        del X_test_batch, batch_data
        
        # Manually trigger garbage collection after each batch
        gc.collect()
            
        batch_time_end = time.time()
        logging.info(f"Batch {batch_start}-{batch_end} processed in {batch_time_end - batch_time_start:.2f} seconds")
    
    total_time = time.time() - start_total
    logging.info(f"Finished calculating AOA values in {total_time:.2f} seconds")
    
    # Create a new result dataset with the original test data and new columns
    result = X_weighted_test.copy()
    
    # Add calculated columns to the result
    result['dissimilarity_index'] = ('dim_0', dissimilarity_indexes)
    result['aoa'] = ('dim_0', aoa_values)
    
    return result

In [ ]:
gc.collect()
clear_memory()

#### Compute DI for each training data and determine AOA across the study area

In [ ]:
# Select necessary columns
band_names = ['clay', 'silt', 'sand', 'rock', 'slope', 'lsf', 'tri', 'twi', 'ndvi_spring', 'ndvi_summer', 'ndvi_autumn', 'bsi_spring', 'bsi_summer', 'bsi_autumn', 'drained', 'landuse_type']
training_df = training_df[band_names]

# Testing data
processed_raster_data = preprocessed_raster.copy()

# Create binary dummy variables for landuse type
dummy_training_data = convert_to_binary_columns(training_df, 'landuse_type', ['arable', 'forest', 'grassland', 'wetland'])
dummy_raster_data = convert_to_binary_columns(processed_raster_data, 'landuse_type', ['arable', 'forest', 'grassland', 'wetland'])

# Define the columns to standardize
cols_to_standardize = ['clay', 'silt', 'sand', 'rock', 'slope', 'lsf', 'tri', 'twi', 'ndvi_spring', 'ndvi_summer', 'ndvi_autumn', 'bsi_spring', 'bsi_summer', 'bsi_autumn', 'drained','landuse_type_arable', 'landuse_type_forest','landuse_type_grassland','landuse_type_wetland']
custom_scaler = CustomStandardScalerOptimized(cols_to_standardize)

# Fit training data to standardize features
dummy_scaled_training_data = custom_scaler.fit_transform(dummy_training_data)
dummy_scaled_testing_data = custom_scaler.transform(dummy_raster_data)

clear_memory()

# Load SHAP values
shap_values = load(r"baseline_aoa_shap.pkl")

# SHAP values; 
shap_order = ['clay', 'silt', 'sand', 'rock', 'slope', 'lsf', 'tri', 'twi', 'ndvi_spring', 'ndvi_summer', 'ndvi_autumn', 'bsi_spring', 'bsi_summer', 'bsi_autumn', 'drained','landuse_type_arable', 'landuse_type_forest','landuse_type_grassland','landuse_type_wetland']

# Change the order of variables based on SHAP
dummy_scaled_training_data = dummy_scaled_training_data[shap_order]
aligned_dummy_scaled_testing_data= dummy_scaled_testing_data[shap_order]

# Ensure the SHAP values are aligned with the variables in the dataset
shap_values_reshaped = shap_values.flatten()

# Multiply each variable in the dataset by its corresponding SHAP value
weighted_training_data = {}
weighted_testing_data = {}
for idx, var_name in enumerate(dummy_scaled_training_data.data_vars):
    weighted_training_data[var_name] = dummy_scaled_training_data[var_name] * shap_values_reshaped[idx]
    weighted_testing_data[var_name] = aligned_dummy_scaled_testing_data[var_name] * shap_values_reshaped[idx]

# Create a new xarray dataset with the weighted variables
final_training_df = xr.Dataset(weighted_training_data)
final_testing_df = xr.Dataset(weighted_testing_data)

clear_memory()

# Compute mean distance for training data
mean_distance_for_training = calculate_mean_pairwise_distances(final_training_df)
print(mean_distance_for_training)

# Compute Dissimilarity Index (DI) for training data
threshold_di = compute_dissimilarity_indexes(final_training_df, mean_distance_for_training)

# # Identify data within the threshold for test data (AOA)
result = compute_and_identify_AOA(X_weighted_train=final_training_df,
                                           X_weighted_test=final_testing_df,
                                           mean_distance_for_training=mean_distance_for_training,
                                           threshold_di=threshold_di)

# End time
end_time = time.time()

# Calculate elapsed time
execution_time = end_time - start_time
print(f"Execution Time: {execution_time} seconds")

In [ ]:
print("Threshold DI: ", threshold_di)
print(result['aoa'].loc[result['aoa'] == 0])
print(result['aoa'].loc[result['aoa'] == 1])

### Save it to the GeoTIFF

In [ ]:
print(result.rio.crs)
print(result.rio.resolution())

In [ ]:
# Define the variables to export
variable_names = ['dissimilarity_index', 'aoa']

# Get the exact grid definition from the original raster
original_x = original_raster.x.values
original_y = original_raster.y.values
original_transform = original_raster.rio.transform()
original_crs = original_raster.rio.crs

# Create a rasterio transform object once
from rasterio.transform import Affine
inv_transform = ~Affine.from_gdal(*original_transform.to_gdal())

# Define grid shape once
grid_shape = (len(original_y), len(original_x))

# For each variable
for var_name in variable_names:
    print(f"Processing {var_name}...")
    
    # Create the output grid with nodata values
    output_grid = np.full(grid_shape, -9999, dtype=np.float32)
    
    # Get coordinates from result dataset - don't load values yet
    x_coords = result.x
    y_coords = result.y
    
    # Process in larger chunks but not too large to overwhelm memory
    # Adjust chunk_size based on your system's available RAM
    chunk_size = 10000000
    
    # Calculate total number of points
    num_points = len(x_coords.values)
    num_chunks = (num_points + chunk_size - 1) // chunk_size
    
    for i in tqdm(range(num_chunks), desc=f"Processing {var_name} in chunks"):
        start_idx = i * chunk_size
        end_idx = min((i + 1) * chunk_size, num_points)
        
        # Load just this chunk of data
        chunk_slice = slice(start_idx, end_idx)
        
        # Load chunk coordinates
        chunk_x = x_coords.values[chunk_slice]
        chunk_y = y_coords.values[chunk_slice]
        
        # Load chunk values - only compute this chunk if using dask
        if isinstance(result[var_name].data, da.Array):
            chunk_values = result[var_name].data[chunk_slice].compute()
        else:
            chunk_values = result[var_name].values[chunk_slice]
        
        # Vectorized calculation of indices for the entire chunk
        col_indices, row_indices = np.array(inv_transform * (chunk_x, chunk_y)).astype(int)
        
        # Filter valid indices
        valid_mask = (
            (row_indices >= 0) & 
            (row_indices < len(original_y)) & 
            (col_indices >= 0) & 
            (col_indices < len(original_x))
        )
        
        # Apply valid mask to indices and values
        valid_rows = row_indices[valid_mask]
        valid_cols = col_indices[valid_mask]
        valid_values = chunk_values[valid_mask]
        
        # Use np.maximum for overlapping points to keep the highest value
        # Or use another strategy like mean, median, etc. depending on your needs
        for j in range(len(valid_rows)):
            output_grid[valid_rows[j], valid_cols[j]] = valid_values[j]
        
        # Clear memory
        del chunk_x, chunk_y, chunk_values, col_indices, row_indices
        
    # Create DataArray with spatial properties
    output_da = xr.DataArray(
        output_grid,
        dims=["y", "x"],
        coords={"x": original_x, "y": original_y}
    )
    
    # Set spatial properties
    output_da.rio.write_transform(original_transform, inplace=True)
    output_da.rio.write_crs(original_crs, inplace=True)
    output_da.rio.write_nodata(-9999, inplace=True)
    
    # Save to GeoTIFF with memory-efficient settings
    output_da.rio.to_raster(
        f'result/baseline_{var_name}.tif',
        driver="GTiff",
        dtype="float32",
        compress="LZW",
        tiled=True,
        blockxsize=512,
        blockysize=512,
        num_threads="all_cpus"
    )
    
    # Clear memory before processing next variable
    del output_grid, output_da
    import gc
    gc.collect()
    
    print(f"Saved {var_name} to estonia_raster_{var_name}.tif")